# NB3 · 蒙特卡洛与随机近似：方差塌缩与步长两条件

对应主站 **L5（蒙特卡洛方法）与 L6（随机近似与 TD）**，难度 level 2，预计 40 分钟。

这本笔记本刻意**不用 GridWorld**——要研究的两个现象（样本均值的方差塌缩、RM 步长的两个收敛条件）只需要一个最简随机环境：**一枚掷出 +1 概率为 0.7 的硬币**。

主线只有一句话：

> 多 seed 亲眼看到 MC 均值的方差按 σ/√n 塌缩 → 把增量均值改写成 Robbins–Monro 求根 → 用三种步长 α=1/t^k 看到"两个条件"各自在数据上的长相。

**前置**（建议先在主站走完这两条推导链，再来跑代码验证）：

- L5 · 推导「MC 均值无偏性 E[x̄]=E[X] 与 var/n 收缩」（l5-qa 节）；
- L6 · 推导「Robbins–Monro：从均值估计到求根；步长条件 Σα=∞、Σα²<∞」（l6-qa 节）。

## 怎么用这本笔记本

- **Shift + Enter** 逐格运行；带 `# TODO(you)` 的格子要**你自己写实现**——只给签名与提示，不给答案；
- 带 ✅ 的自检格用 `assert` 判分，**全绿才算完成**；数据由固定 seed 生成，你与任何同学跑出的数字应当逐位一致；
- 🏔 挑战格是开放题，**没有参考答案**；
- 本本用到 matplotlib（浏览器内首启多下载约 8–10MB，之后有缓存）；浏览器刷新会清空 kernel，全部重跑即可。

## 0 · 实验环境：一枚偏心的硬币

随机变量 X 只取两个值：

$$X = \begin{cases}+1, & \text{概率 } 0.7 \\ -1, & \text{概率 } 0.3\end{cases}$$

两个总体量手算即得（后面处处要用）：

$$\mu = \mathbb{E}[X] = 0.7 - 0.3 = 0.4, \qquad \sigma^2 = \mathbb{E}[X^2] - \mu^2 = 1 - 0.16 = 0.84$$

一次采样 = 一次掷币。MC 的无偏、方差塌缩、步长敏感，全部能在这枚硬币上看清。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

P_HEADS = 0.7                       # P(X = +1)
MU = 2 * P_HEADS - 1                # E[X] = 0.4
SIGMA = float(np.sqrt(1 - MU**2))   # std[X] = sqrt(0.84)

assert abs(MU - 0.4) < 1e-12 and abs(SIGMA - np.sqrt(0.84)) < 1e-12
print(f"mu = {MU},  sigma^2 = {SIGMA**2:.4f},  sigma = {SIGMA:.4f}")

采样函数是**给定的**——全班同学拿到逐位相同的数据，自检格的 assert 才有共同的裁决标准。

In [ ]:
def draw_coins(rng, n):
    """用 rng 掷 n 次偏心硬币：返回 (n,) 数组，元素 ∈ {+1, -1}，P(+1)=P_HEADS。"""
    return ((rng.random(n) < P_HEADS).astype(float)) * 2 - 1

# 快检：10 万次掷币，+1 的频率应贴近 0.7
freq = float((draw_coins(np.random.default_rng(999), 100_000) == 1).mean())
assert abs(freq - P_HEADS) < 0.01
print(f"10 万次掷币频率 = {freq:.4f} ≈ 0.7")

## 1 · 实验一：样本均值的多 seed 方差塌缩

样本均值（前 n 个样本的平均）：

$$\bar{x}_n = \frac{1}{n}\sum_{i=1}^{n} X_i$$

L5 的推导链给过两个结论：**无偏** $\mathbb{E}[\bar{x}_n]=\mu$，以及 **var/n 收缩** $\mathrm{Var}[\bar{x}_n] = \sigma^2/n$——标准差按 σ/√n 塌缩。现在用代码把这两件事**看在眼里**。

### TODO 1 · 单条轨迹

用 0 号种子掷 10000 次硬币，算出 x̄_1, x̄_2, …, x̄_10000 的完整轨迹。

In [ ]:
def cumulative_mean(x):
    """样本数组 → 前缀均值数组：返回 (N,) 数组，第 n 项是 x̄_n。

    提示：np.cumsum 与 np.arange(1, N+1) 一行可得，无需循环。
    """
    # TODO(you): 删掉下面这行 raise，写下你的实现
    raise NotImplementedError("完成 TODO 1：前缀均值轨迹")


N_MAX = 10_000
x0 = draw_coins(np.random.default_rng(0), N_MAX)   # 给定：0 号种子的 10000 次掷币
traj0 = cumulative_mean(x0)                        # (10000,)  x̄_1 .. x̄_10000

print("前 5 个掷币 :", x0[:5].astype(int))
print(f"x̄_10 = {traj0[9]:+.4f}    x̄_10000 = {traj0[-1]:+.4f}   (mu = {MU})")

### 无偏性快检（给定代码）

无偏说的是**期望**：E[x̄_n]=μ。单条轨迹看不见期望——把「n=100 的实验」整体重复 2000 次，这 2000 个 x̄ 的均值应非常接近 μ，而它们的散布应恰为 σ/√100 ≈ 0.092。注意这一格会复用你刚写的 `cumulative_mean`——它同时也在检验你的 TODO 1。

In [ ]:
rng_rep = np.random.default_rng(123)
xbars_100 = np.array([cumulative_mean(draw_coins(rng_rep, 100))[-1] for _ in range(2000)])

print(f"2000 个 x̄_100 的均值   = {xbars_100.mean():.4f}   (mu = {MU})")
print(f"2000 个 x̄_100 的标准差 = {xbars_100.std(ddof=1):.4f}   (理论 sigma/sqrt(100) = {SIGMA/10:.4f})")

assert abs(xbars_100.mean() - MU) < 0.02, "无偏性：2000 次重复的均值应贴近 mu"
assert 0.9 <= xbars_100.std(ddof=1) / (SIGMA / 10) <= 1.1, "散布应贴近 sigma/sqrt(n)"
print("✅ 无偏性成立：E[x̄] ≈ mu；散布 ≈ sigma/sqrt(100)——方差塌缩同款公式")

### TODO 2 · 换 10 个 seed 各跑一条

一个 seed 是一条轨迹、一个故事；**多 seed 才能看见方差**——这正是主站把「多 seed 方差」当作第一公民的原因。对 seeds 0..9 各生成一条 x̄ 轨迹，叠成 (10, 10000) 的数组。

In [ ]:
SEEDS = list(range(10))           # 10 个种子：0..9

def multi_seed_trajectories(seeds, n_max):
    """对每个 seed 用独立的 np.random.default_rng(seed) 抽 n_max 个样本，
    返回形状 (len(seeds), n_max) 的数组：第 i 行是该 seed 的 x̄_1 .. x̄_{n_max} 轨迹。

    提示：循环里两行——抽样本、复用 cumulative_mean。
    """
    # TODO(you): 删掉下面这行 raise，写下你的实现
    raise NotImplementedError("完成 TODO 2：10 个 seed 的轨迹矩阵")


trajs = multi_seed_trajectories(SEEDS, N_MAX)      # (10, 10000)

print("形状 :", trajs.shape)
print("10 个 seed 的 x̄_10000 :", np.round(trajs[:, -1], 4))

In [ ]:
# 图内文字用英文：Pyodide 内核没有中文字体（解读见下方 markdown）
n_axis = np.arange(1, N_MAX + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(n_axis, trajs.T, color="tab:blue", alpha=0.55, lw=1.0)          # 10 条轨迹
ax.axhline(MU, color="crimson", ls="--", lw=1.6, label=r"true $\mu = 0.4$")
ax.plot(n_axis, MU + SIGMA / np.sqrt(n_axis), color="k", ls=":", lw=1.2, label=r"$\mu \pm \sigma/\sqrt{n}$")
ax.plot(n_axis, MU - SIGMA / np.sqrt(n_axis), color="k", ls=":", lw=1.2)
ax.set_xscale("log")
ax.set_xlabel("n (number of samples)")
ax.set_ylabel(r"sample mean $\bar{x}_n$")
ax.set_title(r"10 seeds: sample means funnel into $\mu$")
ax.legend(loc="upper right", frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# ✅ 自检 1a：n=10000 时，10 条轨迹全部收在 mu 附近
final_devs = np.abs(trajs[:, -1] - MU)
print("10 个 seed 的 |x̄_10000 − mu| :", np.round(final_devs, 4))
assert final_devs.max() < 0.05, "n=10000 时每个 seed 都应满足 |x̄ − mu| < 0.05"
print(f"✅ 多 seed 收敛：max |x̄_10000 − mu| = {final_devs.max():.4f} < 0.05")

# ✅ 自检 1b：方差塌缩 std(x̄_n) ≈ sigma/sqrt(n)。
# 只用 10 个 seed 估标准差太抖（采样误差约 24%）——换 200 个 seed（100..299）做正式估计：
trajs200 = np.stack([cumulative_mean(draw_coins(np.random.default_rng(s), N_MAX)) for s in range(100, 300)])
print(f"\n{'n':>8} {'empirical std':>15} {'sigma/sqrt(n)':>15} {'ratio':>7}")
for n in (100, 1000, 10000):
    emp = trajs200[:, n - 1].std(ddof=1)
    theo = SIGMA / np.sqrt(n)
    print(f"{n:>8} {emp:>15.5f} {theo:>15.5f} {emp/theo:>7.3f}")
    assert 0.8 <= emp / theo <= 1.2, f"n={n} 的 std 比值应落在 [0.8, 1.2]"
print("✅ 方差塌缩：经验 std(x̄_n) / (sigma/√n) 全部落在 [0.8, 1.2]")

### 看到了什么

- 10 条轨迹像一束收口的漏斗：前几十个样本时上蹿下跳，n 越大越贴着 μ——这就是 $\mathrm{Var}[\bar{x}_n]=\sigma^2/n$ 的样子；
- 黑色点线 σ/√n 包络正好框住这束轨迹的散布宽度：**理论曲线与实验肉眼一致**；
- 也要看到反方向的事实：**任何单条轨迹都不保证单调靠近 μ**，某条轨迹可能长时间偏在一边。MC 的收敛是"平均意义上"的收敛，方差是它永远要缴的税。

## 2 · 实验二：Robbins–Monro 求根与三种步长

把增量均值改写成求根问题。想解 $g(\theta) = \theta - \mu = 0$（根显然是 μ），但 μ 未知，只能看到噪声观测：

$$\dot{g}(\theta) = \theta - X, \qquad \mathbb{E}[\dot{g}(\theta)] = \theta - \mu = g(\theta)$$

Robbins–Monro 迭代（沿噪声观测的负方向走一步）：

$$\theta_{t+1} = \theta_t - \alpha_t(\theta_t - X_{t+1}) = \theta_t + \alpha_t (X_{t+1} - \theta_t)$$

取 $\alpha_t = 1/t^k$，k ∈ {0.7, 1.0, 1.3}。L6 的推导给过收敛的**两个条件**：

$$\sum_t \alpha_t = \infty \ \ (\text{步长总量够用，走得到底}), \qquad \sum_t \alpha_t^2 < \infty \ \ (\text{噪声总算得清零})$$

三种 k 正好对应三种命运——先用部分和看条件谁成立（给定代码），再跑迭代亲眼看（TODO 3）。

In [ ]:
t_all = np.arange(1, N_MAX + 1)
print(f"{'k':>4} {'Σα (N=1000)':>12} {'Σα (N=10000)':>13}   {'走势':<8} {'Σα² (N=10000)':>14}")
for k in (0.7, 1.0, 1.3):
    a_1000 = float((1.0 / np.arange(1, 1001) ** k).sum())
    a_all = float((1.0 / t_all ** k).sum())
    a2_all = float((1.0 / t_all ** (2 * k)).sum())
    verdict = "增长→∞" if k <= 1 else "已饱和(有界)"
    print(f"{k:>4} {a_1000:>12.2f} {a_all:>13.2f}   {verdict:<8} {a2_all:>14.3f}")

print("\n读表：三种 k 的 Σα² 全部有界（第二条件全过）；")
print("     但 k=1.3 的 Σα 已近饱和——步长总量有限，第一条件被破坏。")

### TODO 3 · 三种步长，同一份数据

对 0 号种子那份数据 `x0`（TODO 1 里生成的）分别用 k=0.7 / 1.0 / 1.3 跑 RM 迭代。**数据完全相同，唯一的变量是步长**——结尾的命运差异就只能是步长造成的。

In [ ]:
def rm_track(x, k, theta0=0.0):
    """RM 求根 g(θ)=θ−μ：θ_{t+1} = θ_t + α_t·(x_{t+1} − θ_t)，α_t = 1/t^k。

    x     : (N,) 掷币样本（噪声源）
    k     : 步长指数
    theta0: 初值
    返回  : (N,) 数组 θ_1, θ_2, …, θ_N（处理完第 t 个样本后的估计）。
    提示：t 从 1 数到 N 的 Python 循环即可，一万步毫秒级。
    """
    # TODO(you): 删掉下面这行 raise，写下你的实现
    raise NotImplementedError("完成 TODO 3：RM 轨迹")


KS = [0.7, 1.0, 1.3]
theta_tracks = {k: rm_track(x0, k) for k in KS}    # 同一份数据 x0，三种步长

for k in KS:
    print(f"k={k}:  θ_10000 = {theta_tracks[k][-1]:+.4f},  |θ_10000 − μ| = {abs(theta_tracks[k][-1] - MU):.4f}")

In [ ]:
# 图内文字用英文：Pyodide 内核没有中文字体（解读见下方 markdown）
t_axis = np.arange(1, N_MAX + 1)
colors = {0.7: "tab:blue", 1.0: "tab:green", 1.3: "tab:red"}

fig, ax = plt.subplots(figsize=(8, 4.5))
for k in KS:
    ax.plot(t_axis, np.abs(theta_tracks[k] - MU), color=colors[k], lw=1.1, alpha=0.9, label=fr"$k={k}$")
ax.plot(t_axis, SIGMA / np.sqrt(t_axis), color="k", ls=":", lw=1.2, label=r"$\sigma/\sqrt{t}$ (level for $k=1$)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("t (RM steps)")
ax.set_ylabel(r"$|\theta_t - \mu|$")
ax.set_title(r"RM with $\alpha_t = 1/t^k$: same data, three step sizes")
ax.legend(loc="lower left", frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# ✅ 自检 2：末端误差取最后 100 步 |θ−μ| 的均值（单点值抖，窗口均值稳）
end_err = {k: float(np.abs(theta_tracks[k][-100:] - MU).mean()) for k in KS}
for k in KS:
    print(f"k={k}: 末端误差（最后 100 步平均）= {end_err[k]:.4f}")

assert end_err[1.0] < end_err[0.7] and end_err[1.0] < end_err[1.3], "k=1.0 应是末端误差最小的步长"
assert end_err[1.3] > 5 * end_err[1.0], "k=1.3 的末端误差应显著大于 k=1.0（步长枯竭）"
print("✅ k=1.0 末端误差最小；k=1.3 显著更差（超过 5 倍）——步长枯竭，停在半路")

# 彩蛋自检：k=1.0 的 RM 恰好就是样本均值（α_t = 1/t 时增量均值 = 普通平均）
gap = float(np.max(np.abs(theta_tracks[1.0] - cumulative_mean(x0))))
assert gap < 1e-9, "alpha_t = 1/t 的 RM 应与样本均值轨迹一致"
print(f"✅ k=1.0 的 RM 轨迹与 x̄_n 轨迹最大差 {gap:.1e}——RM(α=1/t) 就是增量均值")

### 连回站点推导：两条件的可视化

对照 L6 推导链里的两个条件，图上三条曲线各归各位：

| k | Σα=∞ ? | Σα²<∞ ? | 图上的命运 |
|---|--------|---------|-----------|
| 0.7 | 成立（部分和随 N 增长） | 成立（Σ1/t^1.4 有界） | 收敛，但后期拖着噪声尾巴，速率只有约 t^(-0.35) |
| 1.0 | 成立（对数增长） | 成立（Σ1/t² 有界） | 收敛且拿到最优速率 t^(-1/2)——两端同时占住 |
| 1.3 | **不成立**（部分和饱和，总量有限） | 成立 | **步长枯竭**：Σα 有限，估计停在半路，末端误差不再下降 |

两条件是"及格线"：不满足必不收敛（k=1.3）；满足了也只是收敛（k=0.7），要快还得 k=1。这正是站点推导里两条件填空在数据上的长相。

还有一个值得记一辈子的等式：**α_t=1/t 的 RM 就是增量均值**（自检格已验证逐位一致）——L5 的 x̄_n 与 L6 的 RM 是同一件事的两种讲法。

## 🏔 挑战（无参考答案）

图里 k=0.7 是"**前期快、后期慢**"的那条：前 100 步冲得最快，10000 步时却比 k=1.0 差一个量级。

1. **为什么前期快？** 比一比 t=10 时三种步长各是多大，想想大步长在离根很远时的好处；
2. **为什么后期慢？** 提示：有效平均窗长约 1/α_t ≈ t^k 个样本——k=0.7 在 t=10000 时"平均"了约多少个样本？σ/√(有效样本数) 是多少？和 k=1.0 比呢？
3. **两种噪声水平试一试**：把硬币收益放大三倍（下方脚手架 `NOISE_SCALE=3`，此时 μ 与 σ 都放大 3 倍），重跑三种 k——哪条曲线的后期平台被抬高了？抬高约几倍？k=1.0 的后期误差 ∝ σ/√t，k=0.7 的平台又 ∝ 什么？

把观察与解释写成几行 print 或一张对比图——这是留给你的开放题。

In [ ]:
# 🏔 挑战脚手架（选做）：两种噪声水平对比
NOISE_SCALE = 3     # 水平 1：原硬币 (scale=1)；水平 2：收益放大 3 倍 (scale=3)

# 提示：
#   x_hi = NOISE_SCALE * draw_coins(np.random.default_rng(0), N_MAX)  # 噪声放大版数据
#   mu_hi = NOISE_SCALE * MU                                         # 新的真值
#   对同一份 x_hi 重跑 rm_track 的三种 k，画 |θ − mu_hi| 对比图，
#   观察 k=0.7 与 k=1.0 的"后期平台"各被抬高了多少倍。
# TODO(challenge): 写下你的实验与结论
print("🏔 挑战区：跑完两种噪声水平后，回来写下你的解释。")

## 回顾：你刚刚做了什么

- **无偏性**：2000 次重复实验的 x̄ 均值贴近 μ——E[x̄]=E[X] 不是玄学，是平均行为；
- **方差塌缩**：10 条轨迹收成漏斗，200 seed 的经验 std(x̄_n) 与 σ/√n 比值全部落在 [0.8, 1.2]——**方差是 MC 的第一公民**，单看一条轨迹永远看不见它；
- **RM 两条件**：k=1.3 破坏 Σα=∞，步长枯竭停在半路；k=0.7 双条件满足但速率慢；k=1.0 又收敛又最快——**条件管生死，速率看步长**；
- **一个等式**：α_t=1/t 的 RM 与样本均值逐位相同——L5 与 L6 在这里合流。

## 下一步

- **NB4 · Q-learning 复现**（L7）：固定步长 α 与 ε-greedy 上场，10-seed 误差带正是本本「多 seed」方法的第一次实战；
- 回主站复习：L5 推导「MC 无偏与 var/n 收缩」、L6 推导「RM 两条件」——现在你两边都亲手碰过了。